In [ ]:
import subprocess

subprocess.run(["pip", "install", "yfinance", "--quiet"])

In [3]:
import yfinance as yf

df = yf.download('AAPL', start='2020-01-01', end='2024-01-01')
close_prices = df['Close'].values

[*********************100%***********************]  1 of 1 completed


In [4]:
print(close_prices.shape)

(1006, 1)


In [17]:
import numpy as np
import torch

X, Y = [], []
slice_window = 30
for i in range(len(close_prices) - slice_window):
    X.append(close_prices[i:i + slice_window])
    Y.append(close_prices[i + slice_window])

X = np.array(X)
Y = np.array(Y)

X = torch.tensor(X, dtype=torch.float32)
Y = torch.tensor(Y, dtype=torch.float32)

print(X.shape)
print(Y.shape)

split = int(len(X) * 0.8)
x_train, y_train = X[:split], Y[:split]
x_test, y_test = X[split:], Y[split:]

torch.Size([976, 30, 1])
torch.Size([976, 1])


In [18]:
from torch.utils.data import DataLoader, TensorDataset

train_loader = DataLoader(
    dataset=TensorDataset(x_train, y_train),
    batch_size=32,
    shuffle=False
)
test_loader = DataLoader(
    dataset=TensorDataset(x_test, y_test),
    batch_size=32,
    shuffle=False
)

In [14]:
import torch.nn as nn


class PriceLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=1, output_size=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # lstm receive input of shape
        self.ltsm = nn.LSTM(
            input_size=input_size,  # 1 feature: price
            hidden_size=hidden_size,  # số chiều hidden state
            num_layers=num_layers,
            batch_first=True  # input dùng shape (batch, seq, feature)
        )

        self.lc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        """
        x shape: (batch_size, 30, 1)
        """

        # lstm_out shape: (batch_size, seq_len, hidden_size)
        # h_n shape: (num_layers, batch_size, hidden_size)
        # c_n shape: (num_layers, batch_size, hidden_size)
        lstm_out, (h_n, c_n) = self.ltsm(x)

        # Lấy hidden state ở timestep cuối cùng
        # last_hidden shape: (batch_size, hidden_size)
        last_hidden = lstm_out[:, -1, :]

        # output shape: (batch_size, output_size)
        output = self.lc(last_hidden)
        return output

In [ ]:
import torch
import torch.nn as nn


class LSTMCellScratch(nn.Module):
    """Single LSTM cell — one timestep at a time."""

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size

        # All 4 gates packed into one matrix multiplication for efficiency
        # W maps [h_{t-1}, x_t] -> (forget, input, candidate, output)
        self.W = nn.Linear(hidden_size + input_size, 4 * hidden_size)

    def forward(self, x_t, h_prev, c_prev):
        """
        x_t:    (batch, input_size)
        h_prev: (batch, hidden_size)
        c_prev: (batch, hidden_size)
        """
        # Concatenate previous hidden state and current input
        combined = torch.cat([h_prev, x_t], dim=1)  # (batch, hidden+input)

        # One matmul to get all 4 gates, then split
        gates = self.W(combined)  # (batch, 4*hidden)
        f, i, g, o = gates.chunk(4, dim=1)

        f_t = torch.sigmoid(f)        # forget gate
        i_t = torch.sigmoid(i)        # input gate
        g_t = torch.tanh(g)           # candidate cell
        o_t = torch.sigmoid(o)        # output gate

        c_t = f_t * c_prev + i_t * g_t   # update cell state
        h_t = o_t * torch.tanh(c_t)      # update hidden state

        return h_t, c_t


class LSTMScratch(nn.Module):
    """Full LSTM that loops over a sequence using LSTMCellScratch."""

    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = LSTMCellScratch(input_size, hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        """
        x shape: (batch, seq_len, input_size)
        """
        batch_size = x.size(0)

        # Initialize hidden and cell state to zeros
        h = torch.zeros(batch_size, self.hidden_size, device=x.device)
        c = torch.zeros(batch_size, self.hidden_size, device=x.device)

        # Step through each timestep manually
        for t in range(x.size(1)):
            x_t = x[:, t, :]       # (batch, input_size)
            h, c = self.cell(x_t, h, c)

        # Only use the last hidden state to predict
        return self.fc(h)


# Quick sanity check
scratch_model = LSTMScratch(input_size=1, hidden_size=64, output_size=1)
dummy = torch.randn(8, 30, 1)    # batch=8, seq=30, features=1
out = scratch_model(dummy)
print("Output shape:", out.shape)  # expect (8, 1)


In [19]:
model = PriceLSTM(input_size=1, hidden_size=64, num_layers=1, output_size=1)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [20]:
num_epochs = 50
for epoch in range(1, num_epochs + 1):
    model.train()
    total_train_loss = 0.0

    for batch_x, batch_y in train_loader:
        pred = model(batch_x)

        loss = criterion(pred, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    model.eval()
    total_test_loss = 0.0

    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            pred = model(batch_x)
            loss = criterion(pred, batch_y)
            total_test_loss += loss.item()

    avg_test_loss = total_test_loss / len(test_loader)

    if epoch % 5 == 0 or epoch == 1:
        print(
            f"Epoch [{epoch}/{num_epochs}] "
            f"Train Loss: {avg_train_loss:.6f} "
            f"Test Loss: {avg_test_loss:.6f}"
        )


Epoch [1/50] Train Loss: 17739.824238 Test Loss: 31841.556083
Epoch [5/50] Train Loss: 16196.936230 Test Loss: 29579.853795
Epoch [10/50] Train Loss: 14340.812520 Test Loss: 27061.605190
Epoch [15/50] Train Loss: 12550.875176 Test Loss: 24478.688337
Epoch [20/50] Train Loss: 10604.763867 Test Loss: 21651.816685
Epoch [25/50] Train Loss: 9131.308196 Test Loss: 19469.324219
Epoch [30/50] Train Loss: 7844.831204 Test Loss: 17489.143973
Epoch [35/50] Train Loss: 6760.600444 Test Loss: 15775.137695
Epoch [40/50] Train Loss: 5830.238192 Test Loss: 14253.411691
Epoch [45/50] Train Loss: 5025.704003 Test Loss: 12890.168666
Epoch [50/50] Train Loss: 4329.654835 Test Loss: 11665.205776
